In [19]:
from typing_extensions import TypedDict
from dotenv import load_dotenv
from typing import Annotated
from langgraph.graph.message import add_messages
from langgraph.graph import START, END, StateGraph

from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage

In [20]:
load_dotenv()

True

In [21]:
llm = init_chat_model(
    model = "gpt-4.1-mini",
    model_provider = "openai",
    max_tokens = 50
)

In [22]:
class State(TypedDict):
    messages: Annotated[ list, add_messages ]

def chatbot_node(state : State):
    message = state["messages"]
    response = llm.invoke(message)
    return { "message": [response] }

def sample_node(state: State):
    print("\n\n\n inside sample node", state)
    return {"messages": [ "This message got returned from sample node...!" ]}

In [23]:
graph_builder = StateGraph(State)

graph_builder.add_node("ChatBot", chatbot_node)
graph_builder.add_node("SampleNode", sample_node)

graph_builder.add_edge(START, "ChatBot")
graph_builder.add_edge("ChatBot", "SampleNode")
graph_builder.add_edge("SampleNode", END)

In [24]:
graph = graph_builder.compile()

In [25]:
update_state = graph.invoke({
    "messages": [HumanMessage(content="This is an invoked msg...!!")]
})

print("\n\n Updated state: ", update_state)




 inside sample node {'messages': [HumanMessage(content='This is an invoked msg...!!', additional_kwargs={}, response_metadata={}, id='51227976-8af2-47d9-ab52-e02bd58e1010')]}


 Updated state:  {'messages': [HumanMessage(content='This is an invoked msg...!!', additional_kwargs={}, response_metadata={}, id='51227976-8af2-47d9-ab52-e02bd58e1010'), HumanMessage(content='This message got returned from sample node...!', additional_kwargs={}, response_metadata={}, id='697d6d87-7cf0-4981-b2d0-3c32d9286f33')]}
